# Data exploration
## Phase 6 data files
Logging in to Hugging Face, mounting Drive and listing the Phase 6 summarization files in the dataset.


In [2]:
import os
from huggingface_hub import login, get_token, list_repo_files
from google.colab import drive

login()
# Passing the token on so !python scripts can also download the data
os.environ["HF_TOKEN"] = get_token()

drive.mount('/content/drive')

# Listing every file in the dataset that belongs to Phase 6
all_files = list_repo_files("Exploration-Lab/CS-781-Capstone", repo_type="dataset")
phase6_files = [f for f in all_files if "Phase_6" in f]

for f in phase6_files:
    print(f)


Mounted at /content/drive
Phase_6_Summ/Test/test.json
Phase_6_Summ/Train/train.json


## Phase 6 files and format
Downloading the train and test files and checking how each document is stored.


In [ ]:
# Downloading all the Phase 6 files and looking at how they are stored
import json
from huggingface_hub import hf_hub_download

paths = {}
for f in phase6_files:
    name = f.split("/")[-1]
    paths[name] = hf_hub_download("Exploration-Lab/CS-781-Capstone", f, repo_type="dataset")

data = {}
for name in ["train.json", "test.json"]:
    with open(paths[name]) as f:
        data[name] = json.load(f)

    print()
    print(name, type(data[name]), len(data[name]))
    print(str(data[name])[:600])


test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            


train.json <class 'list'> 7030
[{'id': '6712', 'document': 'IVIL Appeal No.768 (NT) of 1977 etc. From the judgment Order dated 9.10.1975 of the Madhya Pradesh High Court in M.C.C. No.144 of 1966. V.A. Bobde, B.R.Aggarwala and U.A.Rana for the Appellant. S.V.Deshpande and S.K.Agnihotri for the Respondents. The Judgment of the Court was delivered by RANGANATH MISHRA, CJ. These are appeals by special leave 252 and are directed against the separate decisions of the Madhya Pradesh High Court in references under the Madhya Pradesh Sales Tax Act. Civil Appeal No.768/77 relates to the assessment period 1951 52, Civil Appeal 539/78 

test.json <class 'list'> 100
[{'id': '4963', 'document': 'Tax Reference Cases Nos. 3 to 5 of 1975 Tax Reference under Section 257 of the Income tax Act, 1961 made by the Income tax Appellate Tribunal, Cuttack Bench, Cuttack P.A. Francis, Champat Rai and Miss A. Subhashini for the Appellant. C.S.S. Rao for the Respondent. The Judgment of the Court was delivered by 

## Text casing and lengths
Checking how many documents are lowercase and how long the documents and summaries are in Phase-1 tokens.


In [ ]:
from tokenizers import Tokenizer

train = data["train.json"]
test = data["test.json"]

print("train keys:", list(train[0].keys()))
print("test keys:", list(test[0].keys()))

# Counting documents that are fully lowercase
lowercase = sum(doc["document"] == doc["document"].lower() for doc in train)
print("lowercase train documents:", lowercase, "of", len(train))

# Measuring the length of the first 500 documents and summaries in Phase-1 tokens
tokenizer = Tokenizer.from_file("/content/drive/MyDrive/CS781/data_p1/tokenizer.json")

doc_lengths = sorted(len(tokenizer.encode(doc["document"]).ids) for doc in train[:500])
sum_lengths = sorted(len(tokenizer.encode(doc["summary"]).ids) for doc in train[:500])
test_lengths = sorted(len(tokenizer.encode(doc["document"]).ids) for doc in test)

for name, lengths in [("train documents", doc_lengths), ("train summaries", sum_lengths), ("test documents", test_lengths)]:
    print(name, "tokens: median", lengths[len(lengths) // 2],
          "90%", lengths[int(0.9 * len(lengths))], "max", lengths[-1])

print()
print(train[0]["summary"][:1000])


train keys: ['id', 'document', 'summary']
test keys: ['id', 'document']
lowercase train documents: 0 of 7030
train documents tokens: median 3686 90% 9081 max 105152
train summaries tokens: median 767 90% 1935 max 11086
test documents tokens: median 3806 90% 11278 max 45736

The appellant, a manufacturer of cement, entered into an agreement with the Cement Manufacturing Company of India Limited, for sale of cement. Under the agreement, the appellant was to sell its cement only through the marketing company, and certain sums would be paid for the cement supplied by the marketing company, which had the discretion to fix the sale price. For the 1950 51, 1951 52 and 1952 53 periods when the appellant was assessed to sales tax for the supply of cement, it maintained at the assessment stage that the transactions were not exigible to sales tax as they were covered under the Explanation to Article 286(1)(a). The first Appellate Authority and the Board of Revenue did not accept the stand of the 

## Where the summary words come from
Checking how many of the summary words are found in the start, the middle and the end of the document, for choosing which part of a long document to keep.


In [4]:
# Using 400 words for each part, which is roughly 512 tokens
first_share = 0
middle_share = 0
last_share = 0

for doc in train[:500]:
    words = doc["document"].lower().split()
    summary_words = set(doc["summary"].lower().split())
    middle = len(words) // 2

    first_share += len(summary_words & set(words[:400])) / len(summary_words)
    middle_share += len(summary_words & set(words[middle - 200:middle + 200])) / len(summary_words)
    last_share += len(summary_words & set(words[-400:])) / len(summary_words)

print("summary words found in the first 400 words:", round(first_share / 500, 3))
print("summary words found in the middle 400 words:", round(middle_share / 500, 3))
print("summary words found in the last 400 words:", round(last_share / 500, 3))


summary words found in the first 400 words: 0.302
summary words found in the middle 400 words: 0.316
summary words found in the last 400 words: 0.356


# Data preparation
## Preparing the Phase 6 data
Making the validation split from train and tokenizing the documents and summaries.


In [5]:
!python /content/drive/MyDrive/CS781/phase6/data_prep.py \
    --out_dir /content/drive/MyDrive/CS781/data_p6 \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json


Tokenizing ...
  train: 6679 documents
  val: 351 documents
  test: 100 documents

Saved to /content/drive/MyDrive/CS781/data_p6


# decoder, LoRA fine-tuning
Training the Phase-1 decoder with LoRA on the attention layers to write the summary after the document.



In [3]:
!python /content/drive/MyDrive/CS781/phase6/train.py --arch decoder --method lora \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p6 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p6/decoder_lora


Using device: cuda
6679 train documents, 100 validation documents used for the loss
16 documents per update, 418 steps per epoch, 418 steps total
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt
Phase-1 weights loaded, missing keys: 0, unused keys: 0
Method: lora, training 0.44M of 110.0M parameters (0.40%)
step 10/418  epoch 1  loss 3.3289  elapsed 0.2 min
step 20/418  epoch 1  loss 3.2835  elapsed 0.4 min
step 30/418  epoch 1  loss 3.2560  elapsed 0.5 min
step 40/418  epoch 1  loss 3.2276  elapsed 0.7 min
step 50/418  epoch 1  loss 3.2396  elapsed 0.8 min
step 60/418  epoch 1  loss 3.2252  elapsed 1.0 min
step 70/418  epoch 1  loss 3.1681  elapsed 1.2 min
step 80/418  epoch 1  loss 3.1883  elapsed 1.3 min
step 90/418  epoch 1  loss 3.1455  elapsed 1.5 min
step 100/418  epoch 1  loss 3.1276  elapsed 1.7 min
  validation loss: 3.1592
step 110/418  epoch 1  loss 3.1454  elapsed 1.9 min
step 120/418  epoch 1  loss 3.1557  elapsed 2.1 min
step 13

Predicting the summaries of 50 validation documents and checking ROUGE-1 and ROUGE-L against the real summaries.
.


In [4]:
!pip install -q rouge-score
!python /content/drive/MyDrive/CS781/phase6/predict.py --split val \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p6/decoder_lora/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p6 \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json \
    --out_dir /content/drive/MyDrive/CS781/submissions_p6/decoder_lora



  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using device: cuda
Loaded decoder (lora) from step 418, validation loss 3.1091
Writing summaries for 50 val documents ...
  10/50 documents done, 1.2 min
  20/50 documents done, 2.5 min
  30/50 documents done, 3.8 min
  40/50 documents done, 5.2 min
  50/50 documents done, 6.5 min
Average generated summary length: 375 words
Average reference summary length: 806 words
Validation ROUGE-1 F1: 0.3811
Validation ROUGE-L F1: 0.2053

First summary:
The respondents were arrested by the police on the ground that the police had committed offences under section 138 of the Criminal Procedure Code, and the police were not guilty of the offence. The police officer was directed to take cognizance of the offences under sections 147, 148, 149, 149 and 149 of the Indian Penal Code. The respondents filed a Criminal Revision Petition before the High Court. The High Court d

Predicting the summaries of the test documents and making the submission file.


In [5]:

!python /content/drive/MyDrive/CS781/phase6/predict.py --split test \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p6/decoder_lora/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p6 \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json \
    --out_dir /content/drive/MyDrive/CS781/submissions_p6/decoder_lora


Using device: cuda
Loaded decoder (lora) from step 418, validation loss 3.1091
Writing summaries for 100 test documents ...
  10/100 documents done, 1.2 min
  20/100 documents done, 2.6 min
  30/100 documents done, 3.9 min
  40/100 documents done, 5.2 min
  50/100 documents done, 6.6 min
  60/100 documents done, 7.9 min
  70/100 documents done, 9.2 min
  80/100 documents done, 10.6 min
  90/100 documents done, 11.9 min
  100/100 documents done, 13.3 min
Average generated summary length: 382 words
100 summaries saved, zip ready: /content/drive/MyDrive/CS781/submissions_p6/decoder_lora/submission.zip

First summary:
The assessee company was a company incorporated under the Income Tax Act, 1961. The assessee was a registered firm. The Income Tax Officer, Bombay, in exercise of the powers conferred by section 2(1) of the Income tax Act, 1922, held that the assessee was entitled to the benefit of section 2 (1) (a) of section 1(1)(a) and (2) of that Act. The High Court held that section 2 of

# convmix, LoRMA fine-tuning
Training only the LoRMA weights inside the Phase-1 convmix layers to write the summary after the document.


In [6]:
!python /content/drive/MyDrive/CS781/phase6/train.py --arch convmix --method lorma \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/convmix/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p6 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p6/convmix_lorma


Using device: cuda
6679 train documents, 100 validation documents used for the loss
16 documents per update, 418 steps per epoch, 418 steps total
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/convmix/best_checkpoint.pt
Phase-1 weights loaded, missing keys: 0, unused keys: 0
Method: lorma, training 0.49M of 105.4M parameters (0.47%)
step 10/418  epoch 1  loss 4.2614  elapsed 0.2 min
step 20/418  epoch 1  loss 4.2194  elapsed 0.3 min
step 30/418  epoch 1  loss 4.1666  elapsed 0.5 min
step 40/418  epoch 1  loss 4.1252  elapsed 0.6 min
step 50/418  epoch 1  loss 4.1439  elapsed 0.8 min
step 60/418  epoch 1  loss 4.1324  elapsed 0.9 min
step 70/418  epoch 1  loss 4.0762  elapsed 1.1 min
step 80/418  epoch 1  loss 4.0817  elapsed 1.2 min
step 90/418  epoch 1  loss 4.0465  elapsed 1.4 min
step 100/418  epoch 1  loss 4.0285  elapsed 1.5 min
  validation loss: 4.0616
step 110/418  epoch 1  loss 4.0265  elapsed 1.8 min
step 120/418  epoch 1  loss 4.0554  elapsed 1.9 min
step 1

Predicting the summaries of 50 validation documents and checking ROUGE-1 and ROUGE-L against the real summaries.


In [7]:
!python /content/drive/MyDrive/CS781/phase6/predict.py --split val \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p6/convmix_lorma/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p6 \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json \
    --out_dir /content/drive/MyDrive/CS781/submissions_p6/convmix_lorma


Using device: cuda
Loaded convmix (lorma) from step 418, validation loss 3.9993
Writing summaries for 50 val documents ...
  10/50 documents done, 1.2 min
  20/50 documents done, 2.4 min
  30/50 documents done, 3.6 min
  40/50 documents done, 4.8 min
  50/50 documents done, 5.9 min
Average generated summary length: 358 words
Average reference summary length: 806 words
Validation ROUGE-1 F1: 0.3628
Validation ROUGE-L F1: 0.1981

First summary:
The appellant was a member of the State of Punjab, and the appellant was not a member. The appellant filed a writ petition in the High Court. The High Court dismissed the writ petition. The trial court dismissed the appeal. The Court, HELD: 1. The State of Maharashtra, who was the appellant in the Court of the High Courts, was not entitled to the benefit of the provisions of the Act. The appeal was dismissed by the High court. The order of the District Judge was dismissed. The District Judge, the High Judge held that the appellant had not been abl

Predicting the summaries of the test documents and making the submission file.


In [8]:
!python /content/drive/MyDrive/CS781/phase6/predict.py --split test \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p6/convmix_lorma/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p6 \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json \
    --out_dir /content/drive/MyDrive/CS781/submissions_p6/convmix_lorma


Using device: cuda
Loaded convmix (lorma) from step 418, validation loss 3.9993
Writing summaries for 100 test documents ...
  10/100 documents done, 1.2 min
  20/100 documents done, 2.4 min
  30/100 documents done, 3.6 min
  40/100 documents done, 4.8 min
  50/100 documents done, 5.9 min
  60/100 documents done, 7.1 min
  70/100 documents done, 8.3 min
  80/100 documents done, 9.5 min
  90/100 documents done, 10.7 min
  100/100 documents done, 11.9 min
Average generated summary length: 378 words
100 summaries saved, zip ready: /content/drive/MyDrive/CS781/submissions_p6/convmix_lorma/submission.zip

First summary:
The assessee, a company, in the case of a company in the State of Punjab, and the Income tax Officer, in his opinion, was not entitled to the tax. The assessee filed a suit for the assessment year 2002 9, but the Income Tax Officer held that the assessee was a company under the Act. The Income tax Act, 1961, was a tenant of the company and was not a tenant. The High Court he

# moe, adapter fine-tuning
Training only the adapters added inside the Phase-1 moe layers to write the summary after the document.


In [9]:
!python /content/drive/MyDrive/CS781/phase6/train.py --arch moe --method adapter \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/moe/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p6 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p6/moe_adapter


Using device: cuda
6679 train documents, 100 validation documents used for the loss
16 documents per update, 418 steps per epoch, 418 steps total
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/moe/best_checkpoint.pt
Phase-1 weights loaded, missing keys: 0, unused keys: 0
Method: adapter, training 1.65M of 195.9M parameters (0.84%)
step 10/418  epoch 1  loss 4.4939  elapsed 0.2 min
step 20/418  epoch 1  loss 4.4592  elapsed 0.3 min
step 30/418  epoch 1  loss 4.4065  elapsed 0.5 min
step 40/418  epoch 1  loss 4.3647  elapsed 0.7 min
step 50/418  epoch 1  loss 4.3691  elapsed 0.8 min
step 60/418  epoch 1  loss 4.3538  elapsed 1.0 min
step 70/418  epoch 1  loss 4.3045  elapsed 1.2 min
step 80/418  epoch 1  loss 4.3091  elapsed 1.3 min
step 90/418  epoch 1  loss 4.2739  elapsed 1.5 min
step 100/418  epoch 1  loss 4.2532  elapsed 1.7 min
  validation loss: 4.1830
step 110/418  epoch 1  loss 4.2511  elapsed 1.9 min
step 120/418  epoch 1  loss 4.2828  elapsed 2.1 min
step 130

Predicting the summaries of 50 validation documents and checking ROUGE-1 and ROUGE-L against the real summaries.


In [10]:
!python /content/drive/MyDrive/CS781/phase6/predict.py --split val \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p6/moe_adapter/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p6 \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json \
    --out_dir /content/drive/MyDrive/CS781/submissions_p6/moe_adapter


Using device: cuda
Loaded moe (adapter) from step 418, validation loss 4.1124
Writing summaries for 50 val documents ...
  10/50 documents done, 1.2 min
  20/50 documents done, 2.4 min
  30/50 documents done, 3.6 min
  40/50 documents done, 4.7 min
  50/50 documents done, 5.9 min
Average generated summary length: 398 words
Average reference summary length: 806 words
Validation ROUGE-1 F1: 0.3728
Validation ROUGE-L F1: 0.2036

First summary:
The appellant was a member of the State of Bihar, and the appellant was appointed as a member in the State. The High Court held that the appellant had been a member and the State Government had no jurisdiction to make the order of the High Court. The appellant filed a writ petition before the High court. The State Government dismissed the appeal. The Court dismissed the writ petition. The appellants filed a suit for the appellant to the High Courts below the State, which was dismissed by the High School of the appellant. The respondent filed a petit

Predicting the summaries of the test documents and making the submission file.


In [11]:
!python /content/drive/MyDrive/CS781/phase6/predict.py --split test \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p6/moe_adapter/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p6 \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json \
    --out_dir /content/drive/MyDrive/CS781/submissions_p6/moe_adapter


Using device: cuda
Loaded moe (adapter) from step 418, validation loss 4.1124
Writing summaries for 100 test documents ...
  10/100 documents done, 1.2 min
  20/100 documents done, 2.4 min
  30/100 documents done, 3.5 min
  40/100 documents done, 4.7 min
  50/100 documents done, 5.9 min
  60/100 documents done, 7.0 min
  70/100 documents done, 8.2 min
  80/100 documents done, 9.4 min
  90/100 documents done, 10.6 min
  100/100 documents done, 11.7 min
Average generated summary length: 404 words
100 summaries saved, zip ready: /content/drive/MyDrive/CS781/submissions_p6/moe_adapter/submission.zip

First summary:
The appellant was a member of the company and the assessee was a tenant of the assessee. The appellant filed a suit for the assessment year 1950 11 and the Income tax Officer, the assessee, who was the assessee 's property, and the appellant was entitled to the compensation. The High Court held that the assessee had no jurisdiction to make the application for the purpose of the 

# encdec, knowledge distillation
Training a smaller encdec student (3 encoder and 3 decoder layers) to learn from the real summaries and from the decoder LoRA model as the teacher.


In [12]:
!python /content/drive/MyDrive/CS781/phase6/distill.py \
    --teacher_ckpt /content/drive/MyDrive/CS781/ckpt_p6/decoder_lora/best_checkpoint.pt \
    --p1_ckpt /content/drive/MyDrive/CS781/ckpt_p1/encdec/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p6 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p6/encdec_distill


Using device: cuda
Student (encdec) loaded, missing keys: 0, unused keys: 54
Teacher: 110.0M parameters, student: 64.7M parameters (59% of the teacher)
16 documents per update, 418 steps per epoch, 418 steps total
step 10/418  epoch 1  loss 3.6975  elapsed 0.2 min
step 20/418  epoch 1  loss 3.4667  elapsed 0.3 min
step 30/418  epoch 1  loss 3.3196  elapsed 0.5 min
step 40/418  epoch 1  loss 3.2097  elapsed 0.7 min
step 50/418  epoch 1  loss 3.1522  elapsed 0.9 min
step 60/418  epoch 1  loss 3.0811  elapsed 1.0 min
step 70/418  epoch 1  loss 3.0349  elapsed 1.2 min
step 80/418  epoch 1  loss 2.9848  elapsed 1.4 min
step 90/418  epoch 1  loss 2.9435  elapsed 1.5 min
step 100/418  epoch 1  loss 2.9130  elapsed 1.7 min
  validation loss: 4.6420
step 110/418  epoch 1  loss 2.8806  elapsed 1.9 min
step 120/418  epoch 1  loss 2.8899  elapsed 2.1 min
step 130/418  epoch 1  loss 2.8718  elapsed 2.3 min
step 140/418  epoch 1  loss 2.8676  elapsed 2.4 min
step 150/418  epoch 1  loss 2.8170  elaps

Predicting the summaries of 50 validation documents and checking ROUGE-1 and ROUGE-L against the real summaries.


In [13]:
!python /content/drive/MyDrive/CS781/phase6/predict.py --split val \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p6/encdec_distill/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p6 \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json \
    --out_dir /content/drive/MyDrive/CS781/submissions_p6/encdec_distill


Using device: cuda
Loaded encdec (full) from step 418, validation loss 4.2503
Writing summaries for 50 val documents ...
  10/50 documents done, 0.2 min
  20/50 documents done, 0.2 min
  30/50 documents done, 0.4 min
  40/50 documents done, 0.5 min
  50/50 documents done, 0.7 min
Average generated summary length: 278 words
Average reference summary length: 806 words
Validation ROUGE-1 F1: 0.3363
Validation ROUGE-L F1: 0.1960

First summary:
The appellant was a Hindu Hindu Hindu State of Bombay. The appellant, who was a public servant, was a person of a person, was not a person. The respondent was a tenant of a suit for a person under section 3(1) of the Act. The High Court held that the appellant was not an offence under section 2(1)(b) of section 2 of the Code of Criminal Procedure, and that the respondent was not entitled to the appellant. The trial court held that he was a right of the appellant and the High Court was not in the High court. The State of India was that the High Court

Predicting the summaries of the test documents and making the submission file.


In [14]:
!python /content/drive/MyDrive/CS781/phase6/predict.py --split test \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p6/encdec_distill/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p6 \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json \
    --out_dir /content/drive/MyDrive/CS781/submissions_p6/encdec_distill


Using device: cuda
Loaded encdec (full) from step 418, validation loss 4.2503
Writing summaries for 100 test documents ...
  10/100 documents done, 0.1 min
  20/100 documents done, 0.3 min
  30/100 documents done, 0.4 min
  40/100 documents done, 0.6 min
  50/100 documents done, 0.7 min
  60/100 documents done, 0.8 min
  70/100 documents done, 1.0 min
  80/100 documents done, 1.1 min
  90/100 documents done, 1.3 min
  100/100 documents done, 1.4 min
Average generated summary length: 292 words
100 summaries saved, zip ready: /content/drive/MyDrive/CS781/submissions_p6/encdec_distill/submission.zip

First summary:
The appellant was a company of a company, and the appellant was the company of the company. The appellant company was a tenant of the assessee company. On the assessment year the assessee was a sum of Rs. 2,000. The assessee was the assessee 's business. The Income tax Officer held that the assessee had been a business of the land. The respondent was a business in the company a